# ChatGLM3 LoRA 8-bit 指令微調（2026 版）

## 學習目標

完成本 notebook 後，你將能夠：

1. 理解 8-bit 量化（LLM.int8()）的原理，以及它與 4-bit（NF4 QLoRA）、16-bit 的適用時機差異
2. 以 `BitsAndBytesConfig` 正確設定 8-bit 量化
3. 掌握 PEFT 前必須呼叫 `prepare_model_for_kbit_training()` 的正確順序
4. 使用 `tokenizer.apply_chat_template()` 實現跨模型可攜的對話格式
5. 以 `trl.SFTTrainer` 完成指令微調，降低錯誤率並提升可讀性

## 前置知識

- 本 notebook 接續 `02-16bits_training/chatglm3_lora_16bit.ipynb`（16-bit LoRA 基礎）
- 閱讀前建議先了解 LoRA 低秩分解的概念

## 銜接說明

- 上游：[02-16bits_training/chatglm3_lora_16bit.ipynb](../02-16bits_training/chatglm3_lora_16bit.ipynb)（16-bit LoRA，無量化）
- 下游：[04-4bits_training/chatglm3_qlora_4bit.ipynb](../04-4bits_training/chatglm3_qlora_4bit.ipynb)（4-bit QLoRA，更激進壓縮）

## VRAM 需求概覽

| 模式 | ChatGLM3-6B 所需 VRAM | 適用場景 |
|------|----------------------|----------|
| 16-bit (bf16) | ~12 GB | A100/H100 全精度訓練 |
| **8-bit (本 notebook)** | **~8 GB** | **RTX 3090/4090，訓練/微調平衡點** |
| 4-bit NF4 (QLoRA) | ~5 GB | RTX 3080/4080，最激進壓縮 |

> 若 VRAM 不足 8 GB，請跳至 `04-4bits_training/chatglm3_qlora_4bit.ipynb`。

In [ ]:
# Cell 0: 版本鎖定（確保 2026 環境一致性）
# 執行一次即可；若已安裝正確版本可略過

%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "trl>=0.12" \
    "peft>=0.13" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.44" \
    "evaluate>=0.4" \
    "safetensors>=0.4" \
    "torch>=2.4"

## Step 1：匯入套件

2026 版的套件匯入關鍵元件：

- **`BitsAndBytesConfig`**：以結構化物件設定量化參數，清楚且可序列化
- **`prepare_model_for_kbit_training`**：kbit 模型在接 PEFT 前必須先呼叫此函式，讓 bitsandbytes 正確處理 gradient checkpointing 與 cast
- **`SFTTrainer` / `SFTConfig`**（trl）：統一的指令微調訓練器，內建 response-only loss 遮罩
- **`set_seed`**：確保訓練可重現

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

set_seed(42)

## Step 2：載入資料集

以 HuggingFace Hub dataset id 載入資料集，任何人 clone repo 即可執行，無本機路徑依賴：

```python
ds = load_dataset("silk-road/alpaca-data-gpt4-chinese", split="train")
```

`load_dataset` 優點：
- 無本機路徑依賴，任何人 clone repo 即可執行
- 內建快取，重複執行不重複下載
- 支援 `batched=True` map，後續處理速度提升 3-5 倍

In [ ]:
# 使用 HF Hub dataset id 取代本機磁碟路徑
# 若環境無法連網，可改用 load_dataset("json", data_files="../data/alpaca_data_zh.json")
ds = load_dataset("silk-road/alpaca-data-gpt4-chinese", split="train")
print(ds)
print(ds[0])

## Step 3：資料集前處理

### 3.1 載入 Tokenizer

`AutoTokenizer` 以 HF Hub model id 取代本機路徑（`d:/Pretrained_models/...`）。
`trust_remote_code=True` 在 ChatGLM3 仍為必要，因為它的 tokenizer 含自定義 Python code。

In [ ]:
MODEL_ID = "THUDM/chatglm3-6b-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
print(tokenizer)
print("EOS token:", tokenizer.eos_token, "| id:", tokenizer.eos_token_id)

### 3.2 apply_chat_template 統一對話格式

#### 為什麼使用 apply_chat_template？

`apply_chat_template` 是 HuggingFace 定義的**跨模型可攜抽象**，同一段程式碼換 model id 即可換模型，且訓練與推論共用同一模板，確保格式一致性：

```python
# 2026 寫法（跨模型通用）
messages = [{"role": "user", "content": query}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

相比模型專屬 API，`apply_chat_template` 的優勢：
- 換模型（如 LLaMA、Qwen）無需重寫前處理邏輯
- 訓練側與推論側共用同一模板，確保格式一致
- 支援多模態訊息（image/audio token）

#### 關於 -100 標籤遮罩（教學補充）

指令微調只計算 **response 部分的 loss**，instruction 部分的 label 設為 -100 讓 CrossEntropy 忽略。
SFTTrainer 內部以 `DataCollatorForCompletionOnlyLM` 自動完成此遮罩，不需手刻。

以下保留一個**底層手刻版**以對照教學，讓你理解 SFTTrainer 在背後做了什麼：

In [ ]:
def process_func(example):
    """Low-level tokenization for pedagogical comparison.

    Demonstrates manual -100 label masking so that loss is computed
    only on the assistant response tokens, not the instruction.
    In production, SFTTrainer handles this automatically.
    """
    MAX_LENGTH = 256

    # Build the instruction prompt using apply_chat_template (portable across models)
    query = "\n".join([example["instruction"], example["input"]]).strip()
    messages = [{"role": "user", "content": query}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # appends the assistant turn header
    )

    # Tokenize instruction and response separately
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(
        "\n" + example["output"] + tokenizer.eos_token, add_special_tokens=False
    )["input_ids"]

    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)
    # -100 masks the instruction part: loss is only computed on response tokens
    labels = [-100] * len(prompt_ids) + response_ids

    # Truncate to MAX_LENGTH
    input_ids = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


# batched=True is omitted here because process_func accesses example fields individually.
# For batched processing, use a formatting_func with SFTTrainer instead (see Step 6).
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
print(tokenized_ds)

In [ ]:
# Sanity check: decode full input
print("=== Full input (decoded) ===")
print(tokenizer.decode(tokenized_ds[1]["input_ids"]))

print("\n=== Response only (non -100 labels) ===")
print(tokenizer.decode([x for x in tokenized_ds[1]["labels"] if x != -100]))

## Step 4：建立模型（8-bit 量化）

### 4.1 量化精度比較：何時選 8-bit？

| 精度 | 技術 | VRAM（6B 模型） | 訓練速度 | 精度損失 | 適用場景 |
|------|------|----------------|----------|----------|----------|
| bf16 / fp16 | 原生半精度 | ~12 GB | 最快 | 無 | A100/H100，精度優先 |
| **int8 (8-bit)** | **LLM.int8()** | **~8 GB** | **略慢（約 20%）** | **極低** | **RTX 3090/4090，平衡點** |
| nf4 (4-bit) | QLoRA / NF4 | ~5 GB | 中等 | 低（<1% perplexity） | RTX 3080/4080，VRAM 緊張 |

**8-bit 的運作原理（LLM.int8()）**：
將 weight 以 int8 儲存（節省記憶體），但 forward pass 計算時動態反量化為 bf16 再進行矩陣乘法（稱為 "mixed-precision decomposition"）。相比 4-bit，精度損失更小，但 VRAM 節省較少。

### 4.2 BitsAndBytesConfig

以結構化 `BitsAndBytesConfig` 物件傳入量化設定，清楚、可序列化，且與其他 `from_pretrained` 參數解耦：

```python
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",          # 自動處理 CPU/disk offload
    torch_dtype=torch.bfloat16, # bf16：動態範圍更大、無 NaN 梯度問題
    use_safetensors=True,       # safetensors：更快載入、無程式碼執行風險
)
```

**為何使用 bf16？**
- bf16 與 fp32 共用相同的指數位數（8 bits），動態範圍一致，梯度不易溢出（NaN）
- 現代 GPU（Ampere 架構起）bf16 Tensor Core 速度與 fp16 相當

**為何使用 safetensors？**
- 載入速度：使用 mmap，比傳統格式快 5-10 倍
- 安全性：純資料格式，無任意程式碼執行風險

### 4.3 prepare_model_for_kbit_training 的正確順序

必須在 `get_peft_model()` **之前**呼叫：

```
from_pretrained (with quantization_config)
  ↓
prepare_model_for_kbit_training()  ← 必須在這裡！
  ↓
get_peft_model(model, lora_config)
```

`prepare_model_for_kbit_training` 的作用：
1. 啟用 gradient checkpointing（節省 VRAM，但略慢）
2. 將 LayerNorm 與 embedding 的 dtype cast 為 bf16，確保梯度正確流動
3. 凍結所有非 LoRA 參數

In [ ]:
# 8-bit quantization config via BitsAndBytesConfig
# VRAM: ~8 GB for ChatGLM3-6B; use 4-bit QLoRA if VRAM < 8 GB
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    # int8 uses LLM.int8() mixed-precision decomposition:
    # weights stored as int8, matmul computed in bf16 dynamically
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",           # handles CPU/disk offload automatically
    torch_dtype=torch.bfloat16, # bf16: larger dynamic range, no NaN gradient risk
    trust_remote_code=True,      # required for ChatGLM3 custom architecture
    use_safetensors=True,        # faster loading, no pickle code execution risk
)

print("Model dtype per layer (first 5):")
for name, param in list(model.named_parameters())[:5]:
    print(f"  {name}: {param.dtype}")

## Step 5：LoRA 設定與 PEFT 模型建立

### 5.1 為何需要 prepare_model_for_kbit_training

量化模型在梯度計算時有特殊需求：int8 weight 不支援直接反向傳播，bitsandbytes 需要先設定好 gradient hook 與 dtype cast。若跳過此步驟，訓練時會出現 `RuntimeError: expected scalar type Float but found Half` 或梯度為 None。

In [ ]:
# Must be called BEFORE get_peft_model()
# Enables gradient checkpointing, casts LayerNorm/embeddings to bf16,
# and freezes all non-LoRA parameters.
model = prepare_model_for_kbit_training(model)

print("Model prepared for kbit training.")

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["query_key_value"],  # ChatGLM3 attention projection
    r=8,                   # LoRA rank: higher = more capacity but more VRAM
    lora_alpha=32,         # scaling factor: effective lr = lora_alpha / r * lr
    lora_dropout=0.1,
    bias="none",
)
print(lora_config)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~0.06% trainable params, ~6B total — LoRA efficiency

## Step 6：SFTTrainer — 指令微調訓練器

### 為什麼用 SFTTrainer？

`SFTTrainer` 是 trl 提供的指令微調專用訓練器，相比手刻 Trainer 有以下優勢：

1. 內建 `DataCollatorForCompletionOnlyLM`，自動只計算 response token 的 loss，無需手刻 -100 遮罩邏輯
2. 以 `formatting_func` 統一套用 `apply_chat_template`，訓練與推論格式保證一致
3. 支援自動 packing（將多筆短對話拼接至 max_seq_length，提升 GPU 利用率）
4. `SFTConfig` 整合了 `TrainingArguments` 所有欄位，加上 SFT 專屬參數

### SFTConfig 重要參數說明

| 參數 | 說明 |
|------|------|
| `bf16=True` | 混合精度訓練（2026 標準） |
| `warmup_ratio=0.1` | 前 10% steps 線性 warmup，防止初期 loss 爆炸 |
| `lr_scheduler_type='cosine'` | cosine 衰減，收斂更穩定 |
| `gradient_accumulation_steps=16` | effective batch = 2 × 16 = 32，在小 VRAM 上模擬大 batch |
| `save_safetensors=True` | 儲存為 safetensors 格式 |
| `optim='adamw_torch_fused'` | fused AdamW，比標準 AdamW 快 ~10%，且修正 L2 regularization 方向 |
| `max_seq_length=256` | SFTTrainer 專屬，控制截斷長度 |

In [ ]:
def formatting_func(example):
    """Apply chat template to format training examples.

    Using apply_chat_template ensures training and inference use
    identical prompt formatting, avoiding distribution shift.
    """
    query = "\n".join([example["instruction"], example["input"]]).strip()
    messages = [
        {"role": "user", "content": query},
        {"role": "assistant", "content": example["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False for training (response already included)
    )

In [ ]:
sft_config = SFTConfig(
    output_dir="./chatglm3-lora-8bit",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch = 2 * 16 = 32
    num_train_epochs=1,
    learning_rate=1e-4,
    bf16=True,                         # bf16 mixed precision (2026 standard)
    warmup_ratio=0.1,                  # 10% linear warmup steps
    lr_scheduler_type="cosine",        # cosine decay for stable convergence
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    max_seq_length=256,                # SFTTrainer-specific truncation
    optim="adamw_torch_fused",         # fused AdamW: faster + correct L2 regularization
    save_safetensors=True,             # save in safetensors format
    seed=42,
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},  # we provide pre-tokenized data
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds.select(range(6000)),
    formatting_func=formatting_func,  # applies chat_template + response-only loss mask
    processing_class=tokenizer,
)

## Step 7：模型訓練

In [ ]:
# Training starts here.
# Expected VRAM usage: ~8 GB for ChatGLM3-6B with 8-bit quantization.
# If OOM, reduce per_device_train_batch_size=1 or switch to 4-bit (see 04-4bits_training/).
trainer.train()

## Step 8：儲存模型

`safe_serialization=True` 確保以 safetensors 格式儲存，比 pytorch_model.bin（pickle）更安全且載入更快。

In [ ]:
# Save the LoRA adapter weights (not the full model)
trainer.model.save_pretrained(
    "./chatglm3-lora-8bit-adapter",
    safe_serialization=True,
)
tokenizer.save_pretrained("./chatglm3-lora-8bit-adapter")

print("LoRA adapter saved to ./chatglm3-lora-8bit-adapter")

### (選用) 推送至 HuggingFace Hub

訓練結束後，以 `push_to_hub` 分享結果並記錄最小 model card 資訊（語言、授權、任務標籤）。

In [ ]:
# Optional: push LoRA adapter to HuggingFace Hub
# Requires: huggingface-cli login (or HF_TOKEN env var)

HUB_MODEL_ID = "your-username/chatglm3-lora-8bit-alpaca-zh"

# Uncomment to push:
# trainer.model.push_to_hub(
#     Hub_MODEL_ID,
#     safe_serialization=True,
#     commit_message="Add ChatGLM3 8-bit LoRA adapter for Alpaca-zh",
# )
# tokenizer.push_to_hub(HUB_MODEL_ID)

print(f"To push: trainer.model.push_to_hub('{HUB_MODEL_ID}', safe_serialization=True)")

## Step 9：模型推論

### apply_chat_template 統一推論格式

推論時同樣使用 `apply_chat_template(..., add_generation_prompt=True)` 格式化 prompt，確保推論時的格式與訓練時完全一致，避免 distribution shift。

```python
messages = [{"role": "user", "content": query}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

In [ ]:
from peft import PeftModel

# Load base model and attach LoRA adapter for inference
# If continuing from training cell, skip re-loading and use trainer.model directly
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    use_safetensors=True,
)
infer_model = PeftModel.from_pretrained(base_model, "./chatglm3-lora-8bit-adapter")
infer_model.eval()
print("Inference model loaded.")

In [ ]:
def chat(model, tokenizer, query: str, max_new_tokens: int = 256) -> str:
    """Run inference using apply_chat_template for portable prompt formatting.

    Replaces model.chat() which is ChatGLM3-specific and fragile.
    apply_chat_template guarantees the same format used during training.
    """
    messages = [{"role": "user", "content": query}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # True for inference: append assistant turn header
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (exclude the prompt)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# Test inference
response = chat(infer_model, tokenizer, "數學考試怎麼考高分？")
print(response)

## 小結

本 notebook 展示了 ChatGLM3 8-bit LoRA 微調的完整流程，關鍵要點：

- **量化設定**：以 `BitsAndBytesConfig(load_in_8bit=True)` 傳入量化參數，結構清楚且可序列化
- **模型載入**：使用 `torch_dtype=torch.bfloat16`（動態範圍大、無 NaN 梯度風險）與 `use_safetensors=True`（快速載入、安全）
- **PEFT 準備順序**：`prepare_model_for_kbit_training()` 必須在 `get_peft_model()` 之前呼叫
- **對話格式**：`apply_chat_template()` 跨模型可攜，訓練與推論共用同一格式
- **訓練器**：`SFTTrainer + formatting_func` 自動處理 response-only loss 遮罩
- **資料來源**：`load_dataset("silk-road/...")` 取代本機路徑，任何人 clone 即可執行
- **儲存格式**：`safe_serialization=True` 輸出 safetensors 格式
- **可重現性**：`set_seed(42)` 與 `TrainingArguments(seed=42)` 確保結果可重現

## 練習題

1. **量化比較**：將本 notebook 的 `load_in_8bit=True` 改為 `load_in_4bit=True` 並加上 `bnb_4bit_quant_type='nf4'`，觀察 VRAM 使用量與訓練 loss 的差異。（提示：對照 `04-4bits_training/chatglm3_qlora_4bit.ipynb`）

2. **LoRA rank 實驗**：分別設定 `r=4`、`r=8`、`r=16`，比較 trainable parameters 百分比與最終 loss 的差異。

3. **格式驗證**：在訓練前，從 `tokenized_ds` 隨機取 5 筆，手動 decode `input_ids` 與非 -100 的 `labels`，驗證 response-only 遮罩是否正確。

## 延伸閱讀

- 下一步：[04-4bits_training/chatglm3_qlora_4bit.ipynb](../04-4bits_training/chatglm3_qlora_4bit.ipynb) — 4-bit QLoRA，更激進壓縮
- 技術文章：[LLM.int8() paper](https://arxiv.org/abs/2208.07339) — 8-bit 量化原理
- TRL 文件：[SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) — 官方 SFTTrainer 用法